# 05 — Scientific Abstract Text Generation

This notebook:

- Loads the trained GPT checkpoint
- Loads the Byte-Level BPE tokenizer
- Generates scientific abstracts from a title and subject
- Saves generated samples for Notebook 06 evaluation

In [1]:
from google.colab import drive

drive.mount(
    "/content/drive"
)

Mounted at /content/drive


In [2]:
import os
import sys
import json
import random

import numpy as np
import torch

In [3]:
PROJECT_PATH = (
    "/content/drive/MyDrive/"
    "Scientific-Abstract-GPT"
)

TOKENIZER_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "bpe_tokenizer",
    "tokenizer.json"
)

BEST_CHECKPOINT_PATH = os.path.join(
    PROJECT_PATH,
    "models",
    "best_model.pt"
)

OUTPUT_FOLDER = os.path.join(
    PROJECT_PATH,
    "outputs"
)

GENERATED_SAMPLES_PATH = os.path.join(
    OUTPUT_FOLDER,
    "generated_samples.json"
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

if PROJECT_PATH not in sys.path:

    sys.path.insert(
        0,
        PROJECT_PATH
    )

print(
    "Project path:",
    PROJECT_PATH
)

Project path: /content/drive/MyDrive/Scientific-Abstract-GPT


In [4]:
required_files = {
    "trained checkpoint": (
        BEST_CHECKPOINT_PATH
    ),
    "tokenizer": (
        TOKENIZER_PATH
    )
}

missing_files = []

for file_name, file_path in (
    required_files.items()
):

    exists = os.path.exists(
        file_path
    )

    print(
        f"{file_name}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    if not exists:

        missing_files.append(
            file_path
        )

if missing_files:

    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(missing_files)
    )

print(
    "\nAll required files are available."
)

trained checkpoint: FOUND
tokenizer: FOUND

All required files are available.


In [5]:
from src.gpt_components import (
    GPTConfig,
    GPTLanguageModel,
    load_tokenizer,
    create_prompt,
    extract_abstract,
    set_seed
)

print(
    "GPT components imported successfully."
)

GPT components imported successfully.


In [6]:
SEED = 42

set_seed(
    SEED
)

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Selected device:",
    device
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Selected device: cuda:0
GPU: Tesla T4


Load Tokenizer

In [7]:
tokenizer = load_tokenizer(
    TOKENIZER_PATH
)

vocab_size = (
    tokenizer.get_vocab_size()
)

special_token_ids = {
    "<TITLE>": (
        tokenizer.token_to_id(
            "<TITLE>"
        )
    ),
    "<SUBJECT>": (
        tokenizer.token_to_id(
            "<SUBJECT>"
        )
    ),
    "<ABSTRACT>": (
        tokenizer.token_to_id(
            "<ABSTRACT>"
        )
    ),
    "<END>": (
        tokenizer.token_to_id(
            "<END>"
        )
    )
}

print(
    "Vocabulary size:",
    f"{vocab_size:,}"
)

print(
    "Special token IDs:"
)

print(
    special_token_ids
)

Vocabulary size: 8,000
Special token IDs:
{'<TITLE>': 2, '<SUBJECT>': 3, '<ABSTRACT>': 4, '<END>': 5}


Load Trained Checkpoint

In [8]:
checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model_config = GPTConfig.from_dict(
    checkpoint[
        "model_config"
    ]
)

model = GPTLanguageModel(
    model_config
).to(device)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()

model_device = next(
    model.parameters()
).device

print(
    "Checkpoint loaded successfully."
)

print(
    "Checkpoint step:",
    checkpoint["step"]
)

print(
    "Checkpoint validation loss:",
    f"{checkpoint['validation_loss']:.4f}"
)

print(
    "Model device:",
    model_device
)

Checkpoint loaded successfully.
Checkpoint step: 3000
Checkpoint validation loss: 3.8935
Model device: cuda:0


In [9]:
if (
    model_config.vocab_size
    != vocab_size
):

    raise ValueError(
        "Checkpoint vocabulary size does not "
        "match tokenizer vocabulary size."
    )

if (
    special_token_ids[
        "<END>"
    ]
    is None
):

    raise ValueError(
        "<END> token was not found."
    )

print(
    "Checkpoint and tokenizer validation passed."
)

Checkpoint and tokenizer validation passed.


Text Generation Function

In [10]:
def generate_abstract(
    title,
    subject,
    max_new_tokens=220,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1
):

    prompt = create_prompt(
        title=title,
        subject=subject
    )

    encoded_prompt = tokenizer.encode(
        prompt
    )

    prompt_ids = torch.tensor(
        encoded_prompt.ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    if (
        prompt_ids.shape[1]
        >= model_config.block_size
    ):

        raise ValueError(
            "Prompt is too long for the "
            "configured context length."
        )

    available_generation_length = (
        model_config.block_size
        - prompt_ids.shape[1]
    )

    actual_max_new_tokens = min(
        max_new_tokens,
        available_generation_length
    )

    with torch.no_grad():

        generated_ids = model.generate(
            input_ids=prompt_ids,
            max_new_tokens=(
                actual_max_new_tokens
            ),
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=(
                repetition_penalty
            ),
            end_token_id=(
                special_token_ids[
                    "<END>"
                ]
            )
        )

    generated_text = tokenizer.decode(
        generated_ids[
            0
        ].detach().cpu().tolist(),
        skip_special_tokens=False
    )

    abstract = extract_abstract(
        generated_text
    ).strip()

    return {
        "title": title,
        "subject": subject,
        "prompt": prompt,
        "generated_text": (
            generated_text
        ),
        "abstract": abstract,
        "prompt_token_count": (
            prompt_ids.shape[1]
        ),
        "generated_token_count": (
            generated_ids.shape[1]
            - prompt_ids.shape[1]
        ),
        "generation_parameters": {
            "max_new_tokens": (
                actual_max_new_tokens
            ),
            "temperature": temperature,
            "top_k": top_k,
            "top_p": top_p,
            "repetition_penalty": (
                repetition_penalty
            )
        }
    }

Generate One Abstract

In [11]:
sample_result = generate_abstract(
    title=(
        "Deep Learning for "
        "Medical Image Classification"
    ),
    subject=(
        "Machine Learning"
    ),
    max_new_tokens=200,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1
)

print(
    "TITLE:"
)

print(
    sample_result["title"]
)

print(
    "\nSUBJECT:"
)

print(
    sample_result["subject"]
)

print(
    "\nGENERATED ABSTRACT:"
)

print(
    sample_result["abstract"]
)

print(
    "\nGENERATED TOKENS:",
    sample_result[
        "generated_token_count"
    ]
)

TITLE:
Deep Learning for Medical Image Classification

SUBJECT:
Machine Learning

GENERATED ABSTRACT:
The potential of artificial intelligence in machine learning has led to the potential of deep learning for data mining, in terms of reliability, and privacy. However, most of their applications are often limited, which may be influenced by the data distribution. We present a new approach that can be applied to this problem as a deep neural network (DNNs) algorithm for both deep learning and the training process. Our method is based on a hybrid network called LDA, that uses this layer and then can learn the data distribution from a set of training data. Moreover, we demonstrate the efficacy of our proposed method to facilitate the development of automated deep learning models in the domain of autonomous driving. Finally, we demonstrate that the proposed approach can achieve better accuracy over the state-of-the-art deep learning methods, but also its advantages over the state-of-the-art

Generate Evaluation Samples

In [12]:
generation_requests = [
    {
        "title": (
            "Deep Learning for "
            "Medical Image Classification"
        ),
        "subject": (
            "Machine Learning"
        )
    },
    {
        "title": (
            "Transformer Models for "
            "Scientific Document Summarization"
        ),
        "subject": (
            "Computation and Language"
        )
    },
    {
        "title": (
            "Reinforcement Learning for "
            "Autonomous Robot Navigation"
        ),
        "subject": (
            "Artificial Intelligence"
        )
    },
    {
        "title": (
            "Neural Networks for "
            "Natural Language Understanding"
        ),
        "subject": (
            "Computation and Language"
        )
    },
    {
        "title": (
            "Explainable Artificial Intelligence "
            "for Healthcare Decision Support"
        ),
        "subject": (
            "Artificial Intelligence"
        )
    },
    {
        "title": (
            "Self-Supervised Representation "
            "Learning from Unlabeled Data"
        ),
        "subject": (
            "Machine Learning"
        )
    }
]

generated_samples = []

for sample_number, request in enumerate(
    generation_requests,
    start=1
):

    result = generate_abstract(
        title=request["title"],
        subject=request["subject"],
        max_new_tokens=200,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.1
    )

    generated_samples.append(
        result
    )

    print(
        f"Sample {sample_number} generated."
    )

Sample 1 generated.
Sample 2 generated.
Sample 3 generated.
Sample 4 generated.
Sample 5 generated.
Sample 6 generated.


Display Generated Samples

In [13]:
for sample_number, sample in enumerate(
    generated_samples,
    start=1
):

    print(
        "=" * 80
    )

    print(
        f"SAMPLE {sample_number}"
    )

    print(
        "=" * 80
    )

    print(
        "Title:",
        sample["title"]
    )

    print(
        "Subject:",
        sample["subject"]
    )

    print(
        "\nAbstract:"
    )

    print(
        sample["abstract"]
    )

    print(
        "\nGenerated tokens:",
        sample[
            "generated_token_count"
        ]
    )

    print()

SAMPLE 1
Title: Deep Learning for Medical Image Classification
Subject: Machine Learning

Abstract:
Time series prediction is a significant challenge in the development of intelligent system. A key limitation in real-world machine learning, research in the field of social media is to be in the way to improve the performance of the prediction model. In this work, we propose an online multi-domain model based on time series classification (BI), which learns a sequence to learn from a number of data. This enables efficient analysis for the presence of the prediction of different types of features. We show that a classifier can be used as an optimization problem, as well as many applications of the models. Moreover, we introduce a novel approach for evaluating the quality of the classifiers on the data set. The proposed method improves the accuracy by using some state-of-the-art methods on the WiNet dataset.

Generated tokens: 162

SAMPLE 2
Title: Transformer Models for Scientific Document

Save Generated Samples

In [14]:
generation_output = {
    "checkpoint_path": (
        BEST_CHECKPOINT_PATH
    ),
    "checkpoint_step": (
        checkpoint["step"]
    ),
    "checkpoint_validation_loss": (
        checkpoint[
            "validation_loss"
        ]
    ),
    "model_config": (
        model_config.to_dict()
    ),
    "number_of_samples": (
        len(generated_samples)
    ),
    "samples": (
        generated_samples
    )
}

with open(
    GENERATED_SAMPLES_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        generation_output,
        file,
        indent=2,
        ensure_ascii=False
    )

print(
    "Generated samples saved:"
)

print(
    GENERATED_SAMPLES_PATH
)

Generated samples saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/outputs/generated_samples.json


In [15]:
with open(
    GENERATED_SAMPLES_PATH,
    "r",
    encoding="utf-8"
) as file:

    saved_generation_output = (
        json.load(file)
    )

saved_samples = (
    saved_generation_output[
        "samples"
    ]
)

non_empty_abstracts = sum(
    1
    for sample in saved_samples
    if sample[
        "abstract"
    ].strip()
)

print(
    "Generated samples file:",
    "FOUND"
    if os.path.exists(
        GENERATED_SAMPLES_PATH
    )
    else "MISSING"
)

print(
    "Saved sample count:",
    len(saved_samples)
)

print(
    "Non-empty abstracts:",
    non_empty_abstracts
)

if (
    len(saved_samples)
    != len(generation_requests)
):

    raise RuntimeError(
        "Saved sample count is incorrect."
    )

if non_empty_abstracts == 0:

    raise RuntimeError(
        "All generated abstracts are empty."
    )

Generated samples file: FOUND
Saved sample count: 6
Non-empty abstracts: 6


Final Completion Check

In [16]:
print(
    "=" * 70
)

print(
    "TEXT GENERATION COMPLETED SUCCESSFULLY"
)

print(
    "=" * 70
)

print(
    "Checkpoint used:"
)

print(
    BEST_CHECKPOINT_PATH
)

print(
    "\nGenerated samples:"
)

print(
    len(saved_samples)
)

print(
    "\nEvaluation input file:"
)

print(
    GENERATED_SAMPLES_PATH
)


TEXT GENERATION COMPLETED SUCCESSFULLY
Checkpoint used:
/content/drive/MyDrive/Scientific-Abstract-GPT/models/best_model.pt

Generated samples:
6

Evaluation input file:
/content/drive/MyDrive/Scientific-Abstract-GPT/outputs/generated_samples.json
